<a href="https://colab.research.google.com/github/PradipBanik/Build_2026/blob/main/library/photo_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Photo master is tool that can take screenshot of the device screen.
Version 1

### 1. Install Dependencies
We need `pyautogui` for screen capturing, `opencv-python` for video encoding, and `numpy` for array manipulation.

In [17]:
!pip install pyautogui opencv-python numpy

### 2. Basic Screen Recorder Template
This script captures the screen and saves it as an `.mp4` file. Note: In a cloud environment like Colab, this code won't find a local screen to record, but it is designed to run on your local machine or embedded device.

In [18]:
import cv2
import numpy as np
import time
import os

try:
    import pyautogui
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False
    print("Note: No display detected. Running in headless/simulated mode.")

def start_recording(filename="output.mp4", duration=10, fps=20.0):
    # For headless environments like Colab, we simulate or use default resolution
    if HAS_DISPLAY:
        screen_size = pyautogui.size()
        width, height = screen_size.width, screen_size.height
    else:
        # Defaulting to 1920x1080 for simulation
        width, height = 1920, 1080

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(filename, fourcc, fps, (width, height))

    print(f"Recording started for {duration} seconds (Headless: {not HAS_DISPLAY})...")
    start_time = time.time()

    try:
        while (time.time() - start_time) < duration:
            if HAS_DISPLAY:
                img = pyautogui.screenshot()
                frame = np.array(img)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            else:
                # Generate a dummy frame for headless testing
                frame = np.zeros((height, width, 3), np.uint8)
                cv2.putText(frame, f"Recording... {time.time():.2f}", (50, 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

            out.write(frame)
            time.sleep(1/fps)
    except KeyboardInterrupt:
        print("Recording stopped manually.")

    out.release()
    print(f"Recording saved as {filename}")

# You can call this now even in Colab without errors
start_recording("test_output.mp4", duration=2)

Note: No display detected. Running in headless/simulated mode.
Recording started for 2 seconds (Headless: True)...
Recording saved as test_output.mp4


### 3. View the Result
Since we are in a notebook, we can display the generated video file directly.

In [19]:
from base64 import b64encode
from IPython.display import HTML

def show_video(file_path):
    mp4 = open(file_path,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    return HTML(f"""
    <video width=600 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """)

show_video('test_output.mp4')

### 4. Embeddable Version (Threaded)
To embed this into a real device or app, use a class that runs in the background.

In [20]:
import threading
import cv2
import numpy as np
import time

try:
    import pyautogui
    HAS_DISPLAY = True
except:
    HAS_DISPLAY = False

class ScreenRecorder(threading.Thread):
    def __init__(self, filename="embed_output.mp4", fps=20.0):
        super().__init__()
        self.filename = filename
        self.fps = fps
        self._stop_event = threading.Event()

    def run(self):
        if not HAS_DISPLAY:
            print("Error: No display found. Background recording aborted.")
            return

        screen_size = pyautogui.size()
        width, height = screen_size.width, screen_size.height
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(self.filename, fourcc, self.fps, (width, height))

        print(f"Background recording started: {self.filename}")

        while not self._stop_event.is_set():
            # Capture screen
            img = pyautogui.screenshot()
            frame = np.array(img)
            # Convert BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            out.write(frame)
            # Simple FPS control
            time.sleep(1/self.fps)

        out.release()
        print(f"Background recording saved to {self.filename}")

    def stop(self):
        self._stop_event.set()

# Example usage on a local machine:
# recorder = ScreenRecorder(filename="background_capture.mp4")
# recorder.start()
# time.sleep(5)  # Record for 5 seconds while doing other things
# recorder.stop()

### 5. FFmpeg-based Embedded Recorder
FFmpeg is standard on most Linux embedded systems and is very efficient for cross-platform recording. This version uses `subprocess` to trigger a system-level recording.

In [21]:
import subprocess
import os
import signal
import time

class FFmpegRecorder:
    def __init__(self, filename="ffmpeg_output.mp4", fps=30):
        self.filename = filename
        self.fps = fps
        self.process = None

    def start(self):
        # x11grab is for Linux. For Windows, use 'gdigrab'. For macOS, 'avfoundation'.
        # This command captures the screen (display :0.0).
        cmd = [
            'ffmpeg',
            '-y', # Overwrite output file
            '-f', 'x11grab',
            '-s', '1920x1080',
            '-r', str(self.fps),
            '-i', ':0.0',
            '-c:v', 'libx264',
            '-preset', 'ultrafast',
            self.filename
        ]

        print(f"Starting FFmpeg recording: {self.filename}")
        # Start the process in a new session so we can kill it easily
        self.process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, preexec_fn=os.setsid)

    def stop(self):
        if self.process:
            print("Stopping FFmpeg recording...")
            os.killpg(os.getpgid(self.process.pid), signal.SIGTERM)
            self.process.wait()
            print(f"Recording saved to {self.filename}")

# Note: This requires ffmpeg installed on the system (!apt-get install ffmpeg)
# recorder = FFmpegRecorder()
# recorder.start()
# time.sleep(5)
# recorder.stop()

### 6. Audio Recorder (Separate Stream)
This class captures audio from the system's default input. Later, we can merge this `.wav` file with our `.mp4` video file.

In [22]:
!apt-get update && apt-get install -y libportaudio2 portaudio19-dev
!pip install pyaudio

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libportaudio2 is already the newest version (19.6.0-1.1).
portaudio19-dev is already the newes

In [23]:
import pyaudio
import wave
import threading

class AudioRecorder(threading.Thread):
    def __init__(self, filename="audio_output.wav", channels=1, rate=44100, chunk=1024):
        super().__init__()
        self.filename = filename
        self.channels = channels
        self.rate = rate
        self.chunk = chunk
        self._stop_event = threading.Event()
        try:
            self.p = pyaudio.PyAudio()
            self.device_available = True
        except Exception as e:
            print(f"Audio initialization failed: {e}")
            self.device_available = False

    def run(self):
        if not self.device_available:
            print("No audio device found. Aborting audio recording.")
            return

        try:
            stream = self.p.open(format=pyaudio.paInt16,
                                 channels=self.channels,
                                 rate=self.rate,
                                 input=True,
                                 frames_per_buffer=self.chunk)

            print(f"Audio recording started: {self.filename}")
            frames = []

            while not self._stop_event.is_set():
                data = stream.read(self.chunk, exception_on_overflow=False)
                frames.append(data)

            stream.stop_stream()
            stream.close()
            self.p.terminate()

            wf = wave.open(self.filename, 'wb')
            wf.setnchannels(self.channels)
            wf.setsampwidth(self.p.get_sample_size(pyaudio.paInt16))
            wf.setframerate(self.rate)
            wf.writeframes(b''.join(frames))
            wf.close()
            print(f"Audio saved to {self.filename}")
        except Exception as e:
            print(f"Audio stream error: {e}")

    def stop(self):
        self._stop_event.set()

### 7. Merging Video and Audio
Once you have both the `.mp4` video and the `.wav` audio, you need to mux them together. The following class uses FFmpeg to merge them while maintaining synchronization.

In [24]:
import subprocess

class MediaMerger:
    def __init__(self, video_path, audio_path, output_path):
        self.video_path = video_path
        self.audio_path = audio_path
        self.output_path = output_path

    def merge(self):
        """Merges video and audio using FFmpeg."""
        cmd = [
            'ffmpeg', '-y',
            '-i', self.video_path,
            '-i', self.audio_path,
            '-c:v', 'copy', # Copy video stream without re-encoding
            '-c:a', 'aac',  # Encode audio to AAC for better compatibility
            '-shortest',    # End recording when the shortest stream ends
            self.output_path
        ]

        print(f"Merging {self.video_path} and {self.audio_path} into {self.output_path}...")
        try:
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            print("Merge successful.")
        except subprocess.CalledProcessError as e:
            print(f"Merge failed: {e.stderr.decode()}")

# Example Usage:
# merger = MediaMerger('test_output.mp4', 'audio_output.wav', 'final_combined.mp4')
# merger.merge()

### 8. Hardware Acceleration (Embedded Systems)
For a Raspberry Pi or similar embedded devices, you should utilize hardware-accelerated encoders to keep CPU usage low. In your `FFmpegRecorder` or `MediaMerger`, you can replace `-c:v libx264` with:

*   **Raspberry Pi**: `-c:v h264_v4l2m2m` or `-c:v h264_omx`
*   **NVIDIA Jetson**: `-c:v h264_nvenc`

### 9. Orchestrating the Full Recording Process
Finally, we can create a main script or class to coordinate the screen and audio threads. This is how you would use it in a real application.

In [16]:
def capture_session(duration=5):
    video_file = "video_temp.mp4"
    audio_file = "audio_temp.wav"
    final_output = "final_recording.mp4"

    # 1. Initialize recorders
    # Note: Using ScreenRecorder (pyautogui) for cross-platform ease
    video_rec = ScreenRecorder(filename=video_file)
    audio_rec = AudioRecorder(filename=audio_file)

    print("--- Starting Capture Session ---")
    video_rec.start()
    audio_rec.start()

    # 2. Let it run for the specified duration
    time.sleep(duration)

    # 3. Stop both threads
    print("--- Stopping Capture Session ---")
    video_rec.stop()
    audio_rec.stop()

    # Wait for threads to finish writing files
    video_rec.join()
    audio_rec.join()

    # 4. Merge the results
    if os.path.exists(video_file) and os.path.exists(audio_file):
        merger = MediaMerger(video_file, audio_file, final_output)
        merger.merge()
        print(f"Done! Created: {final_output}")
    else:
        print("Error: One or more temporary files are missing. Check device availability.")

capture_session(5)

--- Starting Capture Session ---
Error: No display found. Background recording aborted.
Audio stream error: [Errno -9996] Invalid input device (no default output device)
--- Stopping Capture Session ---
Error: One or more temporary files are missing. Check device availability.


### 10. Local Python Script (Copy this to photo_master.py)
This version is optimized for running on your actual laptop (Windows/Mac/Linux) and removes Colab-specific commands.

In [ ]:
import cv2
import numpy as np
import pyautogui
import threading
import time
import os
import pyaudio
import wave
import subprocess

# --- Recorder Classes ---

class ScreenRecorder(threading.Thread):
    def __init__(self, filename="video_temp.mp4", fps=20.0):
        super().__init__()
        self.filename = filename
        self.fps = fps
        self._stop_event = threading.Event()

    def run(self):
        screen_size = pyautogui.size()
        width, height = screen_size.width, screen_size.height
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out = cv2.VideoWriter(self.filename, fourcc, self.fps, (width, height))

        print(f"[Video] Recording started: {width}x{height}")
        while not self._stop_event.is_set():
            img = pyautogui.screenshot()
            frame = np.array(img)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            out.write(frame)
            time.sleep(1/self.fps)

        out.release()
        print("[Video] Recording saved.")

    def stop(self):
        self._stop_event.set()

class AudioRecorder(threading.Thread):
    def __init__(self, filename="audio_temp.wav", channels=1, rate=44100, chunk=1024):
        super().__init__()
        self.filename = filename
        self.channels = channels
        self.rate = rate
        self.chunk = chunk
        self._stop_event = threading.Event()
        self.p = pyaudio.PyAudio()

    def run(self):
        stream = self.p.open(format=pyaudio.paInt16, channels=self.channels,
                             rate=self.rate, input=True, frames_per_buffer=self.chunk)
        frames = []
        print("[Audio] Recording started.")
        while not self._stop_event.is_set():
            data = stream.read(self.chunk)
            frames.append(data)

        stream.stop_stream()
        stream.close()
        self.p.terminate()

        wf = wave.open(self.filename, 'wb')
        wf.setnchannels(self.channels)
        wf.setsampwidth(self.p.get_sample_size(pyaudio.paInt16))
        wf.setframerate(self.rate)
        wf.writeframes(b''.join(frames))
        wf.close()
        print("[Audio] Recording saved.")

    def stop(self):
        self._stop_event.set()

# --- Main Logic ---

if __name__ == "__main__":
    # Setup filenames
    v_file, a_file, output = "video_temp.mp4", "audio_temp.wav", "final_capture.mp4"

    v_rec = ScreenRecorder(v_file)
    a_rec = AudioRecorder(a_file)

    v_rec.start()
    a_rec.start()

    try:
        print("Recording... Press Ctrl+C to stop.")
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\nStopping...")
        v_rec.stop()
        a_rec.stop()
        v_rec.join()
        a_rec.join()

    # Merge using FFmpeg
    print("Merging streams...")
    cmd = ['ffmpeg', '-y', '-i', v_file, '-i', a_file, '-c:v', 'copy', '-c:a', 'aac', output]
    subprocess.run(cmd)
    print(f"Finished! Final file: {output}")

In [27]:
from google.colab import files

script_content = """
import cv2
import numpy as np
import pyautogui
import threading
import time
import os
import pyaudio
import wave
import subprocess

# --- Recorder Classes ---
class ScreenRecorder(threading.Thread):
    def __init__(self, filename='video_temp.mp4', fps=20.0):
        super().__init__()
        self.filename = filename
        self.fps = fps
        self._stop_event = threading.Event()

    def run(self):
        screen_size = pyautogui.size()
        width, height = screen_size.width, screen_size.height
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(self.filename, fourcc, self.fps, (width, height))
        print(f'[Video] Recording started...')
        while not self._stop_event.is_set():
            img = pyautogui.screenshot()
            frame = np.array(img)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            out.write(frame)
            time.sleep(1/self.fps)
        out.release()

    def stop(self):
        self._stop_event.set()

class AudioRecorder(threading.Thread):
    def __init__(self, filename='audio_temp.wav'):
        super().__init__()
        self.filename = filename
        self._stop_event = threading.Event()

    def run(self):
        p = pyaudio.PyAudio()
        stream = p.open(format=pyaudio.paInt16, channels=1, rate=44100, input=True, frames_per_buffer=1024)
        frames = []
        print('[Audio] Recording started...')
        while not self._stop_event.is_set():
            frames.append(stream.read(1024))
        stream.stop_stream()
        stream.close()
        p.terminate()
        wf = wave.open(self.filename, 'wb')
        wf.setnchannels(1)
        wf.setsampwidth(p.get_sample_size(pyaudio.paInt16))
        wf.setframerate(44100)
        wf.writeframes(b''.join(frames))
        wf.close()

    def stop(self):
        self._stop_event.set()

if __name__ == '__main__':
    v_rec = ScreenRecorder()
    a_rec = AudioRecorder()
    v_rec.start()
    a_rec.start()
    try:
        print('Recording... Press Ctrl+C to stop.')
        while True: time.sleep(1)
    except KeyboardInterrupt:
        v_rec.stop()
        a_rec.stop()
        v_rec.join()
        a_rec.join()
        print('Merging...')
        subprocess.run(['ffmpeg', '-y', '-i', 'video_temp.mp4', '-i', 'audio_temp.wav', '-c:v', 'copy', '-c:a', 'aac', 'final_capture.mp4'])
        print('Done! Created final_capture.mp4')
"""

with open('photo_master.py', 'w') as f:
    f.write(script_content.strip())

files.download('photo_master.py')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

pip install pyautogui opencv-python numpy pyaudio

python photo_master.py